# Notebook 02 — Lógica Híbrida de Cruzamento: Spatial Join + Fuzzy String Matching

## Contextualização na Dissertação

Este notebook implementa o **núcleo metodológico** da dissertação: o cruzamento entre a base do CNEFE 2022 (IBGE) e o BHMap Endereços (PBH). O problema de *address matching* — a vinculação de registros de endereço provenientes de fontes heterogêneas — é reconhecido na literatura de GIScience como um desafio de natureza **multimodal**, que envolve simultaneamente incerteza posicional e ambiguidade textual (DAVIS JR. et al., 2011; GOLDBERG et al., 2007).

### Abordagem Híbrida

Em vez de depender exclusivamente de similaridade textual (suscetível a homônimos) ou de proximidade espacial (insuficiente em áreas de alta densidade), adotamos uma **lógica híbrida em dois estágios**:

1. **Filtragem espacial** — *Spatial Join* via R-Tree com raio de busca de 50 metros, gerando candidatos
2. **Desambiguação textual** — Fuzzy String Matching (Levenshtein normalizado) para calcular o **MCI** (*Matching Certainty Indicator*)

O MCI, adaptado de Davis Jr. et al. (2011), quantifica a certeza do pareamento em uma escala contínua $[0, 1]$, onde:

$$MCI = \frac{\sum_{i=1}^{n} w_i \cdot sim(a_i, b_i)}{\sum_{i=1}^{n} w_i}$$

em que $sim(a_i, b_i)$ é a razão de Levenshtein normalizada entre os campos $a_i$ (CNEFE) e $b_i$ (BHMap), e $w_i$ são pesos que refletem a importância relativa de cada componente do endereço (logradouro, número, bairro).


In [1]:
import sys
import os
from pathlib import Path

# Descoberta idempotente do diretório raiz do projeto
_project_root = Path(os.path.abspath('')).parent if Path(os.path.abspath('')).name == 'notebooks' else Path(os.path.abspath(''))
if os.getcwd() != str(_project_root):
    os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
%load_ext autoreload
%autoreload 2
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm
tqdm.pandas()
from src import config
from src.log_config import logger
from src.matching import get_spatial_candidates, calculate_mci, resolve_best_match

## 1. Carregamento dos Dados Geoespaciais Processados

Consumimos os GeoParquets produzidos pelo NB01, com geometria nativa em EPSG:31983 (SIRGAS 2000 / UTM 23S). A projeção UTM é essencial para que as operações de distância euclidianas sejam computadas em **metros**, e não em graus — condição necessária para a definição do raio de busca espacial (ZANDBERGEN, 2008).


In [2]:
logger.info("Carregando bases GeoParquet do UTM 23S")
gdf_cnefe_bh = gpd.read_parquet(config.CNEFE_PROCESSED_FILE)
gdf_bhmap = gpd.read_parquet(config.BHMAP_PROCESSED_FILE)

logger.info("Amostra dos schemas para certificar que STD_ está presente")
display(gdf_cnefe_bh[['std_logradouro_completo']].head(2))
display(gdf_bhmap[['std_tipo_logradouro', 'std_nome_logradouro', 'std_numero']].head(2))

2026-03-05T19:01:06.340301Z [info     ] Carregando bases GeoParquet do UTM 23S [__main__]


2026-03-05T19:01:10.225437Z [info     ] Amostra dos schemas para certificar que STD_ está presente [__main__]


,std_logradouro_completo
0,"RUA DIALOGITA, 135"
1,"AVENIDA DOM PEDRO II, 3687"


,std_tipo_logradouro,std_nome_logradouro,std_numero
0,BEC,SERVIDAO,
1,AVE,PERIMETRAL,


## 2. Busca Espacial de Candidatos — Indexação R-Tree

O primeiro estágio do cruzamento utiliza o **índice espacial R-Tree** (GUTTMAN, 1984) para identificar, para cada ponto do CNEFE, todos os endereços do BHMap situados em um raio de 50 metros. A escolha desse raio reflete um compromisso empírico entre:

- **Cobertura** — capturar correspondências legítimas mesmo com pequenos deslocamentos posicionais
- **Precisão** — minimizar o número de candidatos espúrios em áreas de alta densidade viária

Essa estratégia supera a limitação do *string matching* puro, que é cego à localização geográfica. Conforme demonstrado por Goldberg et al. (2007), a combinação de filtro espacial com avaliação textual reduz significativamente as taxas de falsos positivos em processos de geocodificação reversa.

> **Nota técnica:** A complexidade assintótica da busca R-Tree é $O(n \log n)$ para construção e $O(\log n + k)$ por consulta (onde $k$ é o número de candidatos retornados), o que viabiliza o processamento de mais de 1 milhão de registros em tempo razoável.


In [3]:
logger.info("Inicializando motor R-Tree via pygeos/rtree")
df_candidates = get_spatial_candidates(gdf_cnefe_bh, gdf_bhmap, max_distance=50.0)

logger.info("Estatísticas dos vizinhos obtidos:")
display(df_candidates[['id_cnefe', 'id_bhmap', 'spatial_distance']].head(5))

2026-03-05T19:01:10.380894Z [info     ] Inicializando motor R-Tree via pygeos/rtree [__main__]


2026-03-05T19:01:10.382914Z [info     ] Starting Spatial Join (sjoin_nearest) [__main__] max_distance=50.0


2026-03-05T19:01:23.715005Z [info     ] Spatial Join completed         [__main__] total_candidates=1185811


2026-03-05T19:01:23.820531Z [info     ] Estatísticas dos vizinhos obtidos: [__main__]


,id_cnefe,id_bhmap,spatial_distance
0,0,361897.0,7.805313
1,1,143365.0,3.983400
2,2,303950.0,7.045431
3,3,126869.0,3.329680
4,4,182309.0,2.320929


## 3. Avaliação de Incerteza — Fuzzy String Matching e MCI

Para cada par candidato $(c_i, b_j)$ — onde $c_i$ é um registro CNEFE e $b_j$ um candidato BHMap dentro do raio de 50m — calculamos a **similaridade textual normalizada** usando a distância de Levenshtein (LEVENSHTEIN, 1966). O *Matching Certainty Indicator* (MCI) é então derivado como a média ponderada das similaridades por componente:

| Componente | Campo CNEFE | Campo BHMap | Peso ($w_i$) |
|---|---|---|---|
| Logradouro completo | `std_logradouro_completo` | `std_logradouro_completo` | Alto |
| Número | `std_numero` | `std_numero` | Médio |
| Bairro | `std_bairro` | `std_bairro` | Baixo |

A interpretação do MCI segue os limiares:

| Faixa MCI | Classificação | Interpretação |
|---|---|---|
| $\geq 0.80$ | Alta certeza | Pares confiáveis para análise de acurácia posicional |
| $[0.50, 0.80)$ | Certeza moderada | Requerem inspeção; possível ambiguidade toponímica |
| $< 0.50$ | Baixa certeza | Alta probabilidade de pareamento incorreto |
| $= 0.00$ | Órfão | Nenhum candidato encontrado no raio de busca |


In [4]:
logger.info("Aplicando avaliador de Incerteza (MCI) na matriz de O(N)...")

# O MCI varia de 0.0 (totalmente diferente foneticamente) a 1.0 (match semântico perfeito na mesma calçada)
# NOTA: Dependendo do número de candidatos, esta etapa pode demorar alguns minutos.
# O `apply` interno da função usa Pandas otimizado. ProgressBar integrada não reflete `apply` de série, apenas o log final.

df_candidates = calculate_mci(df_candidates)

display(df_candidates[['std_logradouro_completo', 'std_nome_logradouro', 'spatial_distance', 'MCI']].sample(5))

2026-03-05T19:01:23.904259Z [info     ] Aplicando avaliador de Incerteza (MCI) na matriz de O(N)... [__main__]


2026-03-05T19:01:23.905266Z [info     ] Calculating Fuzzy String Matching (MCI) [__main__]


2026-03-05T19:01:28.683449Z [info     ] Fuzzy logic (MCI) calculation completed [__main__]


,std_logradouro_completo,std_nome_logradouro,spatial_distance,MCI
276000,"RUA TRINDADE, 191",TRINDADE,1.282187,0.857143
301665,"RUA CASUARINA, 94",CASUARINA,4.579172,0.896552
1034587,"RUA ONDINA PEDROSA NAHAS, SN",ONDINA PEDROSA NAHAS,2.893613,0.941176
889112,"RUA BENJAMIM QUADROS, 124",BENJAMIM QUADROS,14.652709,0.909091
1012684,"RUA BASILIO DA GAMA, 53",BASILIO DA GAMA,7.165190,0.926829


## 4. Resolução Desambiguadora (1:N → 1:1)

Como a busca espacial pode retornar múltiplos candidatos BHMap para um mesmo ponto CNEFE, aplicamos uma **resolução gulosa** que seleciona, para cada registro CNEFE, o candidato com maior MCI. Em caso de empate, a menor distância euclidiana funciona como critério de desempate.

Esta etapa é análoga ao *record linkage* determinístico descrito na literatura de integração de bases governamentais (CHRISTEN, 2012), adaptada ao domínio geoespacial pela incorporação da componente de proximidade.

### Distribuição dos Resultados

A tabela abaixo sintetiza os resultados do cruzamento e constitui a **linha de base quantitativa** para os notebooks de análise subsequentes.


In [5]:
gdf_matched = resolve_best_match(df_candidates)

# Count final statistics
total_points = len(gdf_cnefe_bh)
perfect_matches = len(gdf_matched[gdf_matched['MCI'] == 1.0])
good_matches = len(gdf_matched[(gdf_matched['MCI'] >= 0.75) & (gdf_matched['MCI'] < 1.0)])
weak_matches = len(gdf_matched[(gdf_matched['MCI'] > 0) & (gdf_matched['MCI'] < 0.75)])
orphans = len(gdf_matched[gdf_matched['MCI'] == 0.0])

profile_stats = pd.DataFrame([{
    'Perfect Matches (MCI = 1.0)': perfect_matches,
    'High Certainty (0.75 <= MCI < 1.0)': good_matches,
    'Low Certainty (0.0 < MCI < 0.75)': weak_matches,
    'Lost / Unmatched (Orphans > 50m)': orphans,
    'Total Originais CNEFE': total_points
}])

display(profile_stats)

2026-03-05T19:01:28.942116Z [info     ] Resolving best matches from candidates [__main__]


2026-03-05T19:01:30.500331Z [info     ] Best match resolution completed [__main__] final_rows=1180102


,Perfect Matches (MCI = 1.0),High Certainty (0.75 <= MCI < 1.0),Low Certainty (0.0 < MCI < 0.75),Lost / Unmatched (Orphans > 50m),Total Originais CNEFE
0,0,987820,174766,17516,1180102


## 5. Exportação da Base Cruzada

O GeoDataFrame resultante — contendo, para cada endereço do CNEFE, o candidato BHMap associado, o MCI, e a distância posicional em metros — é serializado em GeoParquet para consumo nos notebooks de análise (NB03–NB06).

### Síntese Metodológica

A lógica híbrida implementada neste notebook produz uma base cruzada que permite avaliar simultaneamente:

- **Completude** — proporção de endereços CNEFE que encontram par no BHMap (NB03)
- **Acurácia posicional** — erro em metros entre pares de alta certeza (NB04)
- **Incerteza espacial** — autocorrelação e clustering do erro posicional (NB05)
- **Fatores socioespaciais** — relação entre qualidade e vulnerabilidade urbana (NB06)

A abordagem adotada é consonante com a perspectiva de Davis Jr. et al. (2011), que advoga por métodos de avaliação que integrem as dimensões **posicional**, **atributiva** e **contextual** da qualidade de dados geoespaciais.


In [6]:
MATCHED_FILE = config.PROCESSED_DATA_DIR / "cnefe_match_bhmap.parquet"
gdf_matched.to_parquet(MATCHED_FILE)
logger.info("Matriz de Similaridades consolidada em disco.", arquivo=str(MATCHED_FILE))

STATS_FILE = config.TABLES_DIR / "02_match_profile.csv"
profile_stats.to_csv(STATS_FILE, index=False)
logger.info("Tabela CSV de Match Profiling exportada.", arquivo=str(STATS_FILE))

2026-03-05T19:01:35.855896Z [info     ] Matriz de Similaridades consolidada em disco. [__main__] arquivo=C:\Users\mateu\OneDrive\Documentos\UFMG\Mestrado\geocoding-quality-analysis\data\processed\cnefe_match_bhmap.parquet


2026-03-05T19:01:35.860646Z [info     ] Tabela CSV de Match Profiling exportada. [__main__] arquivo=C:\Users\mateu\OneDrive\Documentos\UFMG\Mestrado\geocoding-quality-analysis\outputs\tables\02_match_profile.csv
